In [25]:
from Bio.PDB import PDBParser
import tempfile
from pyrosetta import Pose, pose_from_pdb, init, dump_pdb, get_fa_scorefxn
from math import exp
from random import random
from utils import random_mutation, relax_structure, random_mutation2, fast_relax_structure
from pyrosetta.rosetta.protocols.analysis import InterfaceAnalyzerMover

In [20]:
init()

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python313.Release 2026.28+release.188eabbe2c00638eb408e86cbd67b1fec355b9c7 2026-07-09T14:07:17] retrieved from: http://www.pyrosetta.org
[ WARNING ] Resetting the global tracer options, which have already been set.
[ WARNING ] This will not affect Tracers which

In [15]:
class InteractionError(Exception):
    pass

def interaction(pose, chain1, chain2, pos1, pos2, interaction):
    """Calculate the shortest potential hydrogen-bond distance between two residues.

    :param pose: PyRosetta Pose containing the residues
    :param chain1: PDB chain ID for the first residue
    :param chain2: PDB chain ID for the second residue
    :param pos1: PDB residue position for the first residue
    :param pos2: PDB residue position for the second residue
    :return: shortest distance between potential hydrogen-bonding atoms in Angstroms
    """
    
    # Create a Biopython PDB parser
    parser = PDBParser(QUIET=True)

    # Create a temporary PDB file to store the PyRosetta pose
    with tempfile.NamedTemporaryFile(suffix=".pdb") as tmp:
        # Write the pose to the temporary PDB file
        pose.dump_pdb(tmp.name)

        # Parse the temporary PDB file into a Biopython Structure object
        structure = parser.get_structure(
            "pose",
            tmp.name
        )

    # Get the first model from the structure
    model = structure[0]

    # Retrieve the two chains being analyzed
    chainA = model[chain1]
    chainB = model[chain2]

    # Retrieve the residues at the specified PDB residue positions
    res1 = chainA[pos1]
    res2 = chainB[pos2]

    # Get the three-letter amino acid codes for each residue
    aa1 = res1.get_resname()
    aa2 = res2.get_resname()

    # Get the hydrogen-bonding atoms for each residue
    # The returned atom lists correspond to res1 and res2 respectively
    
    if interaction == 'Hydrogen Bond':
        atoms1, atoms2 = hydrogen_bond_atoms(aa1, aa2)
    elif interaction == 'Salt Bridge':
        atoms1, atoms2 = salt_bridge_atoms(aa1, aa2)
    else:
        raise InteractionError('Interaction must be "Hydrogen Bond" or "Salt Bridge"')

    # Store all possible donor/acceptor atom pair distances
    distance = []

    # Compare every possible hydrogen-bonding atom in res1
    # against every possible hydrogen-bonding atom in res2
    for atom1 in atoms1:
        for atom2 in atoms2:

            # Calculate the distance between the two atoms in Angstroms
            distance.append(res1[atom1] - res2[atom2])

    # Return the shortest distance between any possible
    # hydrogen-bonding atom pair
    return min(distance)

In [16]:
class HydrogenBondError(Exception):
    pass

def hydrogen_bond_atoms(aa1, aa2):
    """Identify potential hydrogen-bonding atoms between two amino acids.

    :param aa1: Three-letter amino acid code for the first residue
    :param aa2: Three-letter amino acid code for the second residue
    :return: lists of hydrogen-bond donor/acceptor atoms corresponding to aa1 and aa2
    """

    # Define atoms that can donate a hydrogen bond for each amino acid
    hbond_donors = {
        "ARG": ["NE", "NH1", "NH2"],
        "ASN": ["ND2"],
        "CYS": ["SG"],
        "GLN": ["NE2"],
        "HIS": ["ND1", "NE2"],
        "LYS": ["NZ"],
        "SER": ["OG"],
        "THR": ["OG1"],
        "TRP": ["NE1"],
        "TYR": ["OH"]
    }

    # Define atoms that can accept a hydrogen bond for each amino acid
    hbond_acceptors = {
        "ASN": ["OD1"],
        "ASP": ["OD1", "OD2"],
        "CYS": ["SG"],
        "GLN": ["OE1"],
        "GLU": ["OE1", "OE2"],
        "HIS": ["ND1", "NE2"],
        "MET": ["SD"],
        "SER": ["OG"],
        "THR": ["OG1"],
        "TYR": ["OH"]
    }

    # Check whether aa1 can donate and aa2 can accept a hydrogen bond
    if (aa1 in hbond_donors and aa2 in hbond_acceptors):
        return hbond_donors[aa1], hbond_acceptors[aa2]

    # Check whether aa2 can donate and aa1 can accept a hydrogen bond
    elif aa2 in hbond_donors and aa1 in hbond_acceptors:
        return hbond_acceptors[aa1], hbond_donors[aa2]

    # Raise an error if neither residue can form a potential hydrogen bond
    else:
        raise HydrogenBondError(
            f'{aa1} and {aa2} cannot make a hydrogen bond'
        )

In [19]:
interaction(pose_from_pdb('pdb_files/7K18_relaxed.pdb'), 'A', 'B', 1612, 15, 'Salt Bridge')

core.import_pose.import_pose: File 'pdb_files/7K18_relaxed.pdb' automatically determined to be of type PDB from contents.
core.conformation.Conformation: [ WARNING ] missing heavyatom:  OXT on residue ALA:CtermProteinFull 100
core.conformation.Conformation: Found disulfide between residues 112 165
core.conformation.Conformation: Found disulfide between residues 116 137
core.conformation.Conformation: Found disulfide between residues 123 147
core.conformation.Conformation: Found disulfide between residues 127 149


np.float32(4.991876)

In [2]:
class SaltBridgeError(Exception):
    pass


def salt_bridge_atoms(aa1, aa2):
    """Identify potential salt-bridge atoms between two amino acids.

    :param aa1: Three-letter amino acid code for the first residue
    :param aa2: Three-letter amino acid code for the second residue
    :return: lists of salt-bridge atoms corresponding to aa1 and aa2
    """

    # Define negatively charged atoms that can participate in salt bridges
    salt_bridge_negative = {
        "ASP": ["OD1", "OD2"],
        "GLU": ["OE1", "OE2"]
    }

    # Define positively charged atoms that can participate in salt bridges
    salt_bridge_positive = {
        "ARG": ["NE", "NH1", "NH2"],
        "LYS": ["NZ"],
        "HIS": ["ND1", "NE2"]
    }

    # Check whether aa1 is positively charged and aa2 is negatively charged
    if (aa1 in salt_bridge_positive and aa2 in salt_bridge_negative):
        return salt_bridge_positive[aa1], salt_bridge_negative[aa2]

    # Check whether aa1 is negatively charged and aa2 is positively charged
    elif (aa1 in salt_bridge_negative and aa2 in salt_bridge_positive):
        return salt_bridge_negative[aa1], salt_bridge_positive[aa2]

    # Raise an error if the residues cannot form a salt bridge
    else:
        raise SaltBridgeError(
            f'{aa1} and {aa2} cannot make a salt bridge'
        )

In [ ]:
class InteractionError(Exception):
    pass

class SaltBridgeError(Exception):
    pass

class HydrogenBondError(Exception):
    pass

class AffinityOptimizerError(Exception):
    pass

class AffinityOptimizer():

    def __init__(self, pdb: str, scoring_function: str, pos_to_mutate: list, temp: int = 1,
                 cooling_rate: float = 0.95, relax: bool = True, relax_every: int = 1,
                 early_stop_iter: int = 30, number_steps: int = 1000):
        """Initializing affinity optimizer

        :param pdb: Path to the PDB file
        :param scoring_function: Scoring function used to evaluate poses
        :param pos_to_mutate: List of residue positions that can be mutated
        :param temp: Initial temperature indicating how exploratory the run function is, defaults to 1
        :param cooling_rate: How quickly the temperature decreases, defaults to 0.95
        :param relax: Set to True to relax the structure at each step, defaults to True
        :param early_stop_iter: Number of steps without improvement before early stopping the run function, defaults to 30
        :param number_steps: Number of steps in the run function, defaults to 1000
        """

        # initialize all input variable
        self.pdb = pdb
        self.current_pose = pose_from_pdb(pdb)
        self.pos_to_mutate = pos_to_mutate
        self.scoring_function = scoring_function
        self.temp = temp
        self.cooling_rate = cooling_rate
        self.relax = relax
        self.relax_every = max(1, relax_every)
        
        self.max_no_improve = early_stop_iter
        self.number_steps = number_steps
        
        # initialize other important variables that will be tracked
        self.scorefxn = get_fa_scorefxn()
        self.iam = None
        self.best_pose = Pose()
        self.best_pose.assign(self.current_pose)
        self.no_improve_steps = 0
        self.best_score = float('inf')
        self.current_score = None

        # if scoring function is selected initialize the following 
        if scoring_function == 'Distance':
            print('Must insert Interactions to optimize')
            print('Example .insert_interaction(1612, 43)')
            self.func = self.__distance
            self.distances = []
            self.chains = []
            self.positions = []
            self.interactions = []
            self.best_distances = []

        # initialize the following for ΔΔG scoring function
        elif scoring_function == 'DDG':
            self.iam = InterfaceAnalyzerMover(1, False, self.scorefxn)
            self.current_score = self.__interface_dg_score(self.current_pose)
            self.best_score = self.current_score
            self.func = self.__ddg  

        # Raise error if invalid scoring function
        else:
            raise AffinityOptimizerError(f'{scoring_function} is not a valid scoring function. Please'
                                         'enter either "DDG" or "Distance"')


    def __interaction_distance(self, chain1, chain2, pos1, pos2, interaction, new_pose=None):
        """Calculate the shortest potential hydrogen-bond distance between two residues.

        :param pose: PyRosetta Pose containing the residues
        :param chain1: PDB chain ID for the first residue
        :param chain2: PDB chain ID for the second residue
        :param pos1: PDB residue position for the first residue
        :param pos2: PDB residue position for the second residue
        :return: shortest distance between potential hydrogen-bonding atoms in Angstroms
        """
        if new_pose is None:
            new_pose = self.current_pose
        
        # Create a Biopython PDB parser
        parser = PDBParser(QUIET=True)

        structure = self.__get_structure(new_pose)

        # Get the first model from the structure
        model = structure[0]

        # Retrieve the two chains being analyzed
        chainA = model[chain1]
        chainB = model[chain2]

        # Retrieve the residues at the specified PDB residue positions
        res1 = chainA[pos1]
        res2 = chainB[pos2]

        # Get the three-letter amino acid codes for each residue
        aa1 = res1.get_resname()
        aa2 = res2.get_resname()

        # Get the hydrogen-bonding atoms for each residue
        # The returned atom lists correspond to res1 and res2 respectively
        
        if interaction == 'Hydrogen Bond':
            atoms1, atoms2 = self.__hydrogen_bond_atoms(aa1, aa2)
        elif interaction == 'Salt Bridge':
            atoms1, atoms2 = self.__salt_bridge_atoms(aa1, aa2)
        else:
            raise InteractionError('Interaction must be "Hydrogen Bond" or "Salt Bridge"')

        # Store all possible donor/acceptor atom pair distances
        distance = []

        # Compare every possible hydrogen-bonding atom in res1
        # against every possible hydrogen-bonding atom in res2
        for atom1 in atoms1:
            for atom2 in atoms2:

                # Calculate the distance between the two atoms in Angstroms
                distance.append(res1[atom1] - res2[atom2])

        # Return the shortest distance between any possible
        # hydrogen-bonding atom pair
        return float(min(distance))


    def __salt_bridge_atoms(self, aa1, aa2):
        """Identify potential salt-bridge atoms between two amino acids.

        :param aa1: Three-letter amino acid code for the first residue
        :param aa2: Three-letter amino acid code for the second residue
        :return: lists of salt-bridge atoms corresponding to aa1 and aa2
        """

        # Define negatively charged atoms that can participate in salt bridges
        salt_bridge_negative = {
            "ASP": ["OD1", "OD2"],
            "GLU": ["OE1", "OE2"]
        }

        # Define positively charged atoms that can participate in salt bridges
        salt_bridge_positive = {
            "ARG": ["NE", "NH1", "NH2"],
            "LYS": ["NZ"],
            "HIS": ["ND1", "NE2"]
        }

        # Check whether aa1 is positively charged and aa2 is negatively charged
        if (aa1 in salt_bridge_positive and aa2 in salt_bridge_negative):
            return salt_bridge_positive[aa1], salt_bridge_negative[aa2]

        # Check whether aa1 is negatively charged and aa2 is positively charged
        elif (aa1 in salt_bridge_negative and aa2 in salt_bridge_positive):
            return salt_bridge_negative[aa1], salt_bridge_positive[aa2]

        # Raise an error if the residues cannot form a salt bridge
        else:
            raise SaltBridgeError(
                f'{aa1} and {aa2} cannot make a salt bridge'
            )


    def __hydrogen_bond_atoms(self, aa1, aa2):
        """Identify potential hydrogen-bonding atoms between two amino acids.

        :param aa1: Three-letter amino acid code for the first residue
        :param aa2: Three-letter amino acid code for the second residue
        :return: lists of hydrogen-bond donor/acceptor atoms corresponding to aa1 and aa2
        """

        # Define atoms that can donate a hydrogen bond for each amino acid
        hbond_donors = {
            "ARG": ["NE", "NH1", "NH2"],
            "ASN": ["ND2"],
            "CYS": ["SG"],
            "GLN": ["NE2"],
            "HIS": ["ND1", "NE2"],
            "LYS": ["NZ"],
            "SER": ["OG"],
            "THR": ["OG1"],
            "TRP": ["NE1"],
            "TYR": ["OH"]
        }

        # Define atoms that can accept a hydrogen bond for each amino acid
        hbond_acceptors = {
            "ASN": ["OD1"],
            "ASP": ["OD1", "OD2"],
            "CYS": ["SG"],
            "GLN": ["OE1"],
            "GLU": ["OE1", "OE2"],
            "HIS": ["ND1", "NE2"],
            "MET": ["SD"],
            "SER": ["OG"],
            "THR": ["OG1"],
            "TYR": ["OH"]
        }

        # Check whether aa1 can donate and aa2 can accept a hydrogen bond
        if (aa1 in hbond_donors and aa2 in hbond_acceptors):
            return hbond_donors[aa1], hbond_acceptors[aa2]

        # Check whether aa2 can donate and aa1 can accept a hydrogen bond
        elif aa2 in hbond_donors and aa1 in hbond_acceptors:
            return hbond_acceptors[aa1], hbond_donors[aa2]

        # Raise an error if neither residue can form a potential hydrogen bond
        else:
            raise HydrogenBondError(
                f'{aa1} and {aa2} cannot make a hydrogen bond'
            )


    def insert_interaction(self, chain1: str, chain2: str, pos1: int, pos2: int, interaction):
        """inserts interaction into the AffinityOptimizer in order to optimize
           distance.

        :param chain1: Chain for first residues Ex: 'A"
        :param chain2: Chain for second residues Ex: 'B'
        :param pos1: Position in chain for first amino acid
        :param pos2: Position in chain for second amino acid
        :param interaction: specify interaction type Ex: 'Salt Bridge' or 'Hydrogen Bond'
        """

        # Find distance for interaction and append to distances
        distance = self.__interaction_distance(chain1, chain2, pos1, pos2, interaction)
        self.distances.append(distance)

        # keep track of chains, positions, and interactions
        self.chains.append([chain1, chain2])
        self.positions.append([pos1, pos2])
        self.interactions.append(interaction)

        # Initialize best distances and best score for run function
        self.best_distances.append(distance)
        self.best_score = self.__distance_score(self.best_distances)


    def __distance_score(self, pose_distances: list):
        """finds score of give distances for poses

        :param pose_distances: distances between interactions
        :return: returns score
        """

        # initialize score
        score = 0

        # find score
        for dis in pose_distances:
            error = abs(dis - 2.8)

            score += error ** 2

        return score


    def find_distances(self, pose: Pose =None):
        """Finds distances of all interactions

        :param pose: pose to find distances, defaults to None
        :return: list of distances
        """

        # if no pose inserted use current pose
        if pose is None:
            pose = self.current_pose

        # initalize distances
        dis = []

        # for each interaction find distance for pose
        for i, (dbl_chain, dbl_pos) in enumerate(zip(self.chains, self.positions)):
            dis.append(self.__interaction_distance(dbl_chain[0], dbl_chain[1], dbl_pos[0], dbl_pos[1], self.interactions[i], pose))

        return dis


    def __distance(self, new_pose: Pose):
        """Calculate the distance score for a new pose and evaluate the pose.

        Calculates the distances between the specified interactions in the new
        pose, then uses the distance score and simulated annealing algorithm to
        determine whether the new pose should be accepted.

        :param new_pose: New pose generated from the mutation step
        """

        new_distances = self.find_distances(new_pose)
        new_score = self.__distance_score(new_distances)
        old_score = self.__distance_score(self.distances)
        self.__algorithm(new_pose, new_score, old_score, new_distances)


    def __algorithm(self, new_pose, new_score, old_score, new_distances=None):
        """Apply simulated annealing acceptance and track the best pose.

        Compares the new score to the old score and determines whether to
        accept the new pose. Better-scoring poses are always accepted, while
        worse-scoring poses may be accepted based on the current temperature.
        Updates the current pose, best pose, scores, distances, and temperature
        as needed.

        :param new_pose: New pose generated from the mutation step
        :param new_score: Score of the new pose
        :param old_score: Score of the current pose before mutation
        :param new_distances: Distances calculated for the new pose, defaults to None
        """

        delta_score = new_score - old_score
        accepted = False
        if delta_score < 0:
            accepted = True
        else:
            prob = exp(-delta_score / self.temp)
            if prob > random():
                accepted = True

        if accepted:
            self.current_pose = new_pose
            if new_distances is not None:
                self.distances = new_distances
            if self.scoring_function == 'DDG':
                self.current_score = new_score

        current_score = self.__distance_score(self.distances) if self.scoring_function == 'Distance' else (self.current_score if accepted else old_score)

        if current_score < self.best_score:
            self.best_pose = Pose()
            self.best_pose.assign(self.current_pose)
            if new_distances is not None:
                self.best_distances = self.distances.copy()
            self.best_score = current_score
            self.no_improve_steps = 0
            print(f'Best pose updated: {self.best_score}')
        else:
            self.no_improve_steps += 1

        self.temp *= self.cooling_rate


    def __interface_dg_score(self, pose: Pose):
        """Calculate the binding affinity of a protein complex.

        Uses the InterfaceAnalyzerMover to calculate the interface binding
        energy of the given pose.

        :param pose: PyRosetta pose containing the protein complex
        :return: Interface binding energy of the complex
        """

        self.iam.apply(pose)
        return self.iam.get_interface_dG()


    def __ddg(self, new_pose: Pose):
        """Calculate the DDG for a new pose and evaluate it.

        Uses the cached score for the current pose and only recomputes the
        full-interface score for the proposed mutant. This avoids redundant
        old-score evaluation and speeds up each step.

        :param new_pose: New pose generated from the mutation step
        """

        new_score = self.__interface_dg_score(new_pose)
        old_score = self.current_score
        self.__algorithm(new_pose, new_score, old_score)

        if new_score < old_score:
            print(f'Improved interface dG: {new_score} (was {old_score})')
        else:
            print(f'Interface dG unchanged/worse: {new_score} (current {old_score})')
        print(f'DDG: {new_score - old_score}')


    def run(self):
        """Run the affinity optimization algorithm.

        The run function randomly mutates residues in the input structure,
        optionally relaxes the mutated structure, and evaluates the resulting
        pose using the selected scoring function. The algorithm will terminate
        early if the score does not improve for the specified number of steps.
        """

        for i in range(self.number_steps):
            mutant_pose, res = random_mutation2(self.current_pose, self.pos_to_mutate)

            if self.relax and (i % self.relax_every == 0):
                mutant_pose = fast_relax_structure(mutant_pose, res)

            self.func(mutant_pose)

            if (i + 1) % max(1, self.number_steps // 20) == 0 or i == self.number_steps - 1:
                progress = (i + 1) / self.number_steps
                filled = int(progress * 50)
                bar = "#" * filled + "-" * (50 - filled)
                print(f"[{bar}] {progress:.0%}")

            if self.max_no_improve == self.no_improve_steps:
                print('EARLY STOP')
                print(f'Score did not improve in {self.no_improve_steps} steps')
                print(f'Terminated algorithm at {i} step')
                break


    def view_interactions(self):
        """Display the amino acid interactions and their distances.
        Prints the amino acids, chains, positions, and calculated distance
        Prints the amino acids, chains, positions, and calculated distance
        for each interaction in the current structure.
        """
        # Get the current structure and first model
        # Get the current structure and first model
        structure = self.__get_structure()
        model = structure[0]
         # Loop through each interaction and display the residues and distance
         # Loop through each interaction and display the residues and distance
        for num, (chain, pos) in enumerate(zip(self.chains, self.positions)):
            aa1 = model[chain[0]][pos[0]].get_resname()
            aa2 = model[chain[1]][pos[1]].get_resname()
            print(f'Interaction {num + 1}')
            print(f'AA: {aa1}, chain: {chain[0]}, pos: {pos[0]}')
            print(f'AA: {aa2}, chain: {chain[1]}, pos: {pos[1]}')
            print(f'Distance: {self.distances[num]}')


    def __get_structure(self, new_pose=None):
        """Convert a PyRosetta pose into a Biopython structure.
        Uses a temporary PDB file to convert the PyRosetta pose into a
        Uses a temporary PDB file to convert the PyRosetta pose into a
        Biopython Structure object.
        """
        # if no pose ios given use current pose
        # if no pose ios given use current pose
        if new_pose is None:
            new_pose = self.current_pose
        # Create a Biopython PDB parser
        # Create a Biopython PDB parser
        parser = PDBParser(QUIET=True)

        # Create a temporary PDB file to store the PyRosetta pose
        with tempfile.NamedTemporaryFile(suffix=".pdb") as tmp:
            # Write the pose to the temporary PDB file
            new_pose.dump_pdb(tmp.name)

            # Parse the temporary PDB file into a Biopython Structure object
            structure = parser.get_structure(
                "pose",
                tmp.name
            )
        return structure 

In [ ]:
aa_mutate = [110, 111, 113, 114, 117, 118, 
             119, 120, 138, 139, 140, 141, 142, 144, 
             145, 146, 148, 158, 159, 160, 161, 162,
             166]

opt = AffinityOptimizer('pdb_files/7K18_AQPMSSSPKET_mutant.pdb', 'DDG', 
                        aa_mutate, temp = 20, cooling_rate = 0.95, number_steps=1_000, 
                        early_stop_iter=100)
# opt.insert_interaction('A', 'B', 1616, 64, 'Salt Bridge')
# opt.insert_interaction('A', 'B', 1612, 15, 'Hydrogen Bond')
# opt.insert_interaction('A', 'B', 1611, 43, 'Hydrogen Bond')
# opt.insert_interaction('A', 'B', 1615, 63, 'Salt Bridge')

opt.run()
dump_pdb(opt.current_pose, 'exp.pdb')



--------------------------------------------------
TASK COMPLETE
Successfully mutated TRP at position 148 to ALA
Orginal Energy -277.5458848932439; New energy: -276.1620802289129
--------------------------------------------------


Relaxed energy: -277.6136010798071
mutated score: -23.315127884692828 original score: -23.260287459010442
Best pose updated: -23.315127884692828
Old interface dG: -23.260287459010442
New interface dG: -23.315127884692828
DDG: -0.054840425682385785

PROGRESS:
[--------------------------------------------------] 0%

--------------------------------------------------
TASK COMPLETE
Successfully mutated GLY at position 142 to TRP
Orginal Energy -277.61360107980636; New energy: 1912.8974412319574
--------------------------------------------------


Relaxed energy: -258.92902457864926
mutated score: -20.96360090076439 original score: -23.315127884692828
Old interface dG: -23.315127884692828
New interface dG: -20.96360090076439
DDG: 2.351526983928437

PROGRESS:
[--

In [12]:
dump_pdb(relax_structure(opt.best_pose), 'exp2.pdb')
opt = AffinityOptimizer('exp.pdb', 'Distance', aa_mutate, relax=False)
opt.insert_interaction('A', 'B', 1616, 64, 'Salt Bridge')
opt.insert_interaction('A', 'B', 1612, 15, 'Hydrogen Bond')
opt.insert_interaction('A', 'B', 1611, 43, 'Hydrogen Bond')
opt.insert_interaction('A', 'B', 1615, 63, 'Salt Bridge')
opt.view_interactions()

Relaxed energy: -242.89012214741143
Must insert Interactions to optimize
Example .insert_interaction(1612, 43)
chains: [['A', 'B'], ['A', 'B'], ['A', 'B'], ['A', 'B']]
positions: [[1616, 64], [1612, 15], [1611, 43], [1615, 63]]
Interaction 1
AA: GLU, chain: A, pos: 1616
AA: LYS, chain: B, pos: 64
Distance: 2.5238869190216064
Interaction 2
AA: SER, chain: A, pos: 1612
AA: HIS, chain: B, pos: 15
Distance: 8.418946266174316
Interaction 3
AA: SER, chain: A, pos: 1611
AA: HIS, chain: B, pos: 43
Distance: 2.9515395164489746
Interaction 4
AA: LYS, chain: A, pos: 1615
AA: GLU, chain: B, pos: 63
Distance: 7.676762104034424


In [ ]:
print("final pose score:", opt.__distance_score(opt.find_distances(opt.current_pose)))
print("best pose score:", opt.__distance_score(opt.best_distances))
print("best distances:", opt.best_distances)
print("final distances:", opt.find_distances(opt.current_pose))

final pose score: 22.810541542581596
best pose score: 22.810541542581596
best distances: [2.788937568664551, 5.325052261352539, 3.339505195617676, 6.817893028259277]
final distances: [2.788937568664551, 5.325052261352539, 3.339505195617676, 6.817893028259277]
